# 17 — Ablation & Hurdle Model, Cross-Period (Setups A and C)

Checks whether two of this project's central findings (from `13_ablation.ipynb` and
`15_Hurdle_Model.ipynb`, both built for the primary Setup B / Q1 2026) also hold in Setup A
(Q4 2025) and Setup C (Feb-Apr 2026):

1. **Ablation ladder** -- does most of the accuracy gain come from engineered features and
   K-fold target encoding (A1 -> A2 -> A3), on top of the model/ensemble choice (XGBoost ->
   blend), in every period, not just Q1 2026?
2. **Naive-vs-clean target** -- does using the clean multi-snapshot target instead of the
   naive single-snapshot count matter in every period? (This is the single largest ΔR²
   contribution in Ali's reference-paper ablation figure: +0.040.)
3. **Hurdle model** -- does the two-stage (zero/nonzero classifier + positive-count
   regressor) framing improve on a single XGBoost model in every period, or was Setup B's
   ~0.5% gain a one-quarter fluke?

Reuses the XGBoost/Blend dev-CV numbers already computed in
`16_Cross_Period_Robustness.ipynb` (`outputs/cross_period_robustness.csv`) rather than
re-fitting them, and only computes the pieces that notebook didn't produce: the A1/A2/A3
ladder, the naive-target comparison, and the hurdle model.

## 1. Setup

In [1]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings('ignore')
OUT = Path('../outputs')
RS, K = 42, 5
TARGET, ID = 'blocked_days_Q1_2026', 'id'

robustness = pd.read_csv(OUT / 'cross_period_robustness.csv')
print(robustness[['setup', 'model', 'dev_mse', 'dev_r2']].to_string(index=False))

setup            model    dev_mse   dev_r2
    A LinearRegression 407.293051 0.534825
    A     RandomForest 332.597387 0.620136
    A          XGBoost 321.169553 0.633188
    A         LightGBM 321.996803 0.632243
    A         CatBoost 325.116909 0.628680
    A    Blend (SLSQP) 317.841688 0.636989
    B LinearRegression 359.997067 0.428628
    B     RandomForest 307.071534 0.512629
    B          XGBoost 294.051664 0.533294
    B         LightGBM 296.494290 0.529417
    B         CatBoost 299.034278 0.525386
    B    Blend (SLSQP) 292.063723 0.536449
    C LinearRegression 380.918645 0.455274
    C     RandomForest 310.554717 0.555897
    C          XGBoost 302.771906 0.567026
    C         LightGBM 306.277933 0.562013
    C         CatBoost 306.097420 0.562271
    C    Blend (SLSQP) 299.724517 0.571384


In [2]:
class KFoldTE(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state
    def _m(self, x, yy):
        st = pd.DataFrame({'c': x, 'y': yy}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.gm_) / (st['count'] + self.smoothing)).to_dict()
    def fit(self, X, y):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        self.maps_ = {c: self._m(X[c].astype(str).fillna('_n'), y) for c in self.cols}; return self
    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = X[c].astype(str).fillna('_n').map(self.maps_[c]).fillna(self.gm_).astype('float32')
        return Xo
    def fit_transform(self, X, y=None, **k):
        y = np.asarray(y, float); self.gm_ = float(y.mean()); Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.gm_, 'float32')
        kk = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kk.split(X):
            for c in self.cols:
                m = self._m(X[c].astype(str).fillna('_n').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = X[c].astype(str).fillna('_n').iloc[va].map(m).fillna(self.gm_).astype('float32').values
        self.maps_ = {c: self._m(X[c].astype(str).fillna('_n'), y) for c in self.cols}; return Xo

## 2. Setup Registry

Per-setup config: feature file, clip bound, and the "daily/history" column-prefix list used
to separate raw static features (A1) from the full engineered set (A2/A3) -- these prefixes
differ per setup since Setup A has no Q3/Q4 split and Setup C adds a `m1_` (January) block.

In [3]:
CROSS_SETUPS = {
    'A': dict(
        train_path=OUT / 'train_local_setupA.parquet',
        naive_path=OUT / 'target_naive_setupA.parquet',
        clip=92, target_label='Q4 2025',
        daily=('hist_', 'm7_', 'm8_', 'm9_', 'recent', 'month', 'weekend', 'weekday',
               'blocking', 'booking', 'available', 'trend', 'delta', 'change', 'cluster', 'geo'),
    ),
    'C': dict(
        train_path=OUT / 'train_local_setupC.parquet',
        naive_path=OUT / 'target_naive_setupC.parquet',
        clip=89, target_label='Feb-Apr 2026',
        daily=('q3_', 'q4_', 'hist_', 'm7_', 'm8_', 'm9_', 'm10_', 'm11_', 'm12_', 'm1_',
               'recent', 'month', 'weekend', 'weekday',
               'blocking', 'booking', 'available', 'trend', 'delta', 'change', 'cluster', 'geo'),
    ),
}
print('Cross-period setups for ablation + hurdle:', list(CROSS_SETUPS))

Cross-period setups for ablation + hurdle: ['A', 'C']


## 3. Ablation Ladder (A1 -> A2 -> A3), per setup

Identical structure to `13_ablation.ipynb`: A1 = Linear Regression on raw static features
only (no daily-history aggregates, no target encoding); A2 = + all engineered features,
high-cardinality categoricals dropped (no target encoding); A3 = + K-fold target encoding
(full preprocessing). All three use the same 5-fold CV split.

In [4]:
def cv_mse(X, y, cols, pp_factory, model_factory, clip, seed=RS, k=K):
    kf = KFold(k, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr, va in kf.split(X):
        pp = pp_factory()
        Xtr = pp.fit_transform(X[cols].iloc[tr], y[tr])
        Xva = pp.transform(X[cols].iloc[va])
        m = model_factory()
        m.fit(Xtr, y[tr])
        oof[va] = np.clip(m.predict(Xva), 0, clip)
    return mean_squared_error(y, oof), r2_score(y, oof)


def is_daily(c, daily_prefixes):
    return any(k in c.lower() for k in daily_prefixes)


ablation_rows = []
naive_rows = []
hurdle_rows = []

for setup_name, cfg in CROSS_SETUPS.items():
    print(f'\n{"="*70}\nSETUP {setup_name} ({cfg["target_label"]}) -- ablation ladder\n{"="*70}')
    train = pd.read_parquet(cfg['train_path'])
    y = train[TARGET].astype(float).values
    X = train.drop(columns=[TARGET, ID], errors='ignore').reset_index(drop=True)
    clip = cfg['clip']

    num_all = X.select_dtypes(include='number').columns.tolist()
    cat_all = X.select_dtypes(exclude='number').columns.tolist()
    raw_num = [c for c in num_all if not is_daily(c, cfg['daily'])]
    card = {c: X[c].nunique(dropna=False) for c in cat_all}
    cat_high = [c for c, n in card.items() if n > 15]
    cat_low  = [c for c, n in card.items() if n <= 15]

    # A1: raw static numeric only, linear
    pp_raw = lambda: ColumnTransformer([('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), raw_num)])
    a1_mse, a1_r2 = cv_mse(X, y, raw_num, pp_raw, LinearRegression, clip)

    # A2: full engineered, NO target encoding (drop high-card cats)
    cols_a2 = num_all + cat_low
    pp_a2 = lambda: ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), num_all),
        ('l', Pipeline([('i', SimpleImputer(strategy='constant', fill_value='m')), ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_low)])
    a2_mse, a2_r2 = cv_mse(X, y, cols_a2, pp_a2, LinearRegression, clip)

    # A3: full pipeline WITH target encoding, linear
    cols_full = num_all + cat_low + cat_high
    pp_full = lambda: ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), num_all),
        ('l', Pipeline([('i', SimpleImputer(strategy='constant', fill_value='m')), ('o', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_low),
        ('h', Pipeline([('te', KFoldTE(cat_high))]), cat_high)])
    a3_mse, a3_r2 = cv_mse(X, y, cols_full, pp_full, LinearRegression, clip)

    xgb_mse = robustness.query("setup == @setup_name and model == 'XGBoost'")['dev_mse'].iloc[0]
    xgb_r2  = robustness.query("setup == @setup_name and model == 'XGBoost'")['dev_r2'].iloc[0]
    blend_mse = robustness.query("setup == @setup_name and model == 'Blend (SLSQP)'")['dev_mse'].iloc[0]
    blend_r2  = robustness.query("setup == @setup_name and model == 'Blend (SLSQP)'")['dev_r2'].iloc[0]

    print(f'A1 raw static linear             MSE={a1_mse:7.2f}  R2={a1_r2:.3f}')
    print(f'A2 + engineered (no TE)           MSE={a2_mse:7.2f}  R2={a2_r2:.3f}   (delta {a2_mse-a1_mse:+.1f})')
    print(f'A3 + K-fold target encoding       MSE={a3_mse:7.2f}  R2={a3_r2:.3f}   (delta {a3_mse-a2_mse:+.1f})')
    print(f'XGBoost (tuned, full preproc)     MSE={xgb_mse:7.2f}  R2={xgb_r2:.3f}   (delta {xgb_mse-a3_mse:+.1f})')
    print(f'SLSQP blend                       MSE={blend_mse:7.2f}  R2={blend_r2:.3f}   (delta {blend_mse-xgb_mse:+.1f})')

    for stage, mse, r2 in [
        ('A1 raw static linear', a1_mse, a1_r2),
        ('A2 + engineered (no TE)', a2_mse, a2_r2),
        ('A3 + K-fold target encoding', a3_mse, a3_r2),
        ('XGBoost (tuned)', xgb_mse, xgb_r2),
        ('SLSQP blend', blend_mse, blend_r2),
    ]:
        ablation_rows.append({'setup': setup_name, 'target': cfg['target_label'], 'stage': stage,
                               'mse': mse, 'r2': r2})


SETUP A (Q4 2025) -- ablation ladder


A1 raw static linear             MSE= 468.68  R2=0.465
A2 + engineered (no TE)           MSE= 407.48  R2=0.535   (delta -61.2)
A3 + K-fold target encoding       MSE= 407.29  R2=0.535   (delta -0.2)
XGBoost (tuned, full preproc)     MSE= 321.17  R2=0.633   (delta -86.1)
SLSQP blend                       MSE= 317.84  R2=0.637   (delta -3.3)

SETUP C (Feb-Apr 2026) -- ablation ladder


A1 raw static linear             MSE= 426.48  R2=0.390
A2 + engineered (no TE)           MSE= 380.79  R2=0.455   (delta -45.7)
A3 + K-fold target encoding       MSE= 380.74  R2=0.456   (delta -0.1)
XGBoost (tuned, full preproc)     MSE= 302.77  R2=0.567   (delta -78.0)
SLSQP blend                       MSE= 299.72  R2=0.571   (delta -3.0)


## 4. Naive-vs-Clean Target, per setup

Fits the same tuned XGBoost config on the SAME features (`X`) but two different targets:
the clean multi-snapshot target (already evaluated in `16_Cross_Period_Robustness.ipynb`,
reused here) vs. the naive single-snapshot count (`target_naive_setup{A,C}.parquet`, saved by
`02b`/`02c` but never otherwise modelled).

**Important caveat, found empirically below, not assumed going in:** neither R² nor absolute
MSE cleanly favors the clean target here. R² is not comparable across two targets with
different variance (R² = 1 - MSE/Var(y)); the naive target has ~1.9-2.3x higher variance than
the clean one (see `02b`/`02c`'s own printed variance ratio), so it shows a *higher* R² below
despite being the target this project deliberately does not model on. Its absolute MSE is
*also* lower here -- which is not evidence the naive target is "easier to predict" in a good
way: a large share of its variance is host-blocking noise that correlates with simple,
structural features (e.g. calendar staleness), so a model can drive both its MSE and R² down
by fitting that noise, without learning anything about actual demand. The justification for
the clean target was never "it scores better by these metrics" -- both notebooks and the
paper are explicit that it is a construct-validity argument (the clean rule isolates a real
available -> booked transition; the naive count conflates that with host-blocking), not a
metrics-superiority one. This section's result does not contradict that; it is exactly the
kind of result the construct-validity argument predicts, and is a useful reminder not to
oversell "our target reduces error" as the reason for the design choice.

In [5]:
xgb_v3_params = {
    'n_estimators': 1000, 'learning_rate': 0.0244, 'max_depth': 6, 'min_child_weight': 11,
    'subsample': 0.769, 'colsample_bytree': 0.740, 'gamma': 0.195,
    'reg_alpha': 0.0615, 'reg_lambda': 4.514, 'tree_method': 'hist', 'n_jobs': -1,
}

def make_pp_simple(num_cols, cat_high):
    return ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median'))]), num_cols),
        ('h', Pipeline([('te', KFoldTE(cat_high))]), cat_high),
    ])

for setup_name, cfg in CROSS_SETUPS.items():
    print(f'\n{"="*70}\nSETUP {setup_name} ({cfg["target_label"]}) -- naive vs. clean target\n{"="*70}')
    train = pd.read_parquet(cfg['train_path'])
    naive_df = pd.read_parquet(cfg['naive_path'])  # columns: id, blocked_days_naive
    clip = cfg['clip']

    merged = train.merge(naive_df, on='id', how='inner')
    y_clean = merged[TARGET].astype(float).values
    y_naive = merged['blocked_days_naive'].astype(float).values
    X = merged.drop(columns=[TARGET, ID, 'blocked_days_naive'], errors='ignore').reset_index(drop=True)

    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_all = X.select_dtypes(exclude='number').columns.tolist()
    card = {c: X[c].nunique(dropna=False) for c in cat_all}
    cat_high = [c for c, n in card.items() if n > 15]

    pp_factory = lambda: make_pp_simple(num_cols, cat_high)
    xgb_ctor = lambda: XGBRegressor(**xgb_v3_params, random_state=RS)

    clean_mse, clean_r2 = cv_mse(X, y_clean, num_cols + cat_high, pp_factory, xgb_ctor, clip)
    naive_mse, naive_r2 = cv_mse(X, y_naive, num_cols + cat_high, pp_factory, xgb_ctor, clip)

    print(f'XGBoost on clean target   MSE={clean_mse:7.2f}  R2={clean_r2:.3f}')
    print(f'XGBoost on naive target   MSE={naive_mse:7.2f}  R2={naive_r2:.3f}')
    print(f'Delta R2 (clean - naive)  = {clean_r2 - naive_r2:+.3f}   <- NOT directly comparable '
          f'to Ali\'s bar (see caveat above): higher naive R2 reflects inflated target variance, '
          f'not better fit.')
    print(f'Delta MSE (naive - clean) = {naive_mse - clean_mse:+.2f}   <- the fair comparison')

    naive_rows.append({'setup': setup_name, 'target': cfg['target_label'],
                        'clean_mse': clean_mse, 'clean_r2': clean_r2,
                        'naive_mse': naive_mse, 'naive_r2': naive_r2,
                        'delta_r2': clean_r2 - naive_r2})


SETUP A (Q4 2025) -- naive vs. clean target


XGBoost on clean target   MSE= 321.29  R2=0.633
XGBoost on naive target   MSE= 314.83  R2=0.807
Delta R2 (clean - naive)  = -0.174   <- NOT directly comparable to Ali's bar (see caveat above): higher naive R2 reflects inflated target variance, not better fit.
Delta MSE (naive - clean) = -6.46   <- the fair comparison

SETUP C (Feb-Apr 2026) -- naive vs. clean target


XGBoost on clean target   MSE= 303.75  R2=0.566
XGBoost on naive target   MSE= 275.32  R2=0.831
Delta R2 (clean - naive)  = -0.265   <- NOT directly comparable to Ali's bar (see caveat above): higher naive R2 reflects inflated target variance, not better fit.
Delta MSE (naive - clean) = -28.44   <- the fair comparison


## 5. Two-Stage Hurdle Model, per setup

Same structure as `15_Hurdle_Model.ipynb`, XGBoost only (not also LightGBM, to keep this
robustness check bounded -- one hurdle model is enough to answer "does the framing help in
other periods"). Classifier `P(y>0|x)` + regressor `E[y|y>0,x]` fit only on positive rows,
combined as `y_hat = clip(P(y>0) * E[y|y>0], 0, clip)`, compared against the plain XGBoost
dev-CV MSE/R² already computed in `16_Cross_Period_Robustness.ipynb`.

In [6]:
XGB_CLF_PARAMS = dict(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=3.0, tree_method='hist',
                     n_jobs=-1, eval_metric='logloss', early_stopping_rounds=50)
XGB_REG_PARAMS = dict(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=3.0, tree_method='hist',
                     n_jobs=-1, eval_metric='rmse', early_stopping_rounds=50)

for setup_name, cfg in CROSS_SETUPS.items():
    print(f'\n{"="*70}\nSETUP {setup_name} ({cfg["target_label"]}) -- hurdle model\n{"="*70}')
    train = pd.read_parquet(cfg['train_path'])
    y = train[TARGET].astype(float).values
    X = train.drop(columns=[TARGET, ID], errors='ignore').reset_index(drop=True)
    clip = cfg['clip']

    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_all = X.select_dtypes(exclude='number').columns.tolist()
    card = {c: X[c].nunique(dropna=False) for c in cat_all}
    cat_high = [c for c, n in card.items() if n > 15]
    print(f'Zero rate: {(y == 0).mean():.3f}')

    kf = KFold(K, shuffle=True, random_state=RS)
    oof = np.zeros(len(y))
    for fold, (tr, va) in enumerate(kf.split(X)):
        pp = make_pp_simple(num_cols, cat_high)
        Xtr_full = pp.fit_transform(X.iloc[tr], y[tr])
        Xva = pp.transform(X.iloc[va])
        ytr = y[tr]
        pos = ytr > 0

        Xtr_c, Xin_c, ytr_c, yin_c = train_test_split(
            Xtr_full, pos.astype(int), test_size=0.10, random_state=RS + fold, stratify=pos)
        Xtr_pos, Xin_pos, ytr_pos, yin_pos = train_test_split(
            Xtr_full[pos], ytr[pos], test_size=0.10, random_state=RS + fold)

        clf = XGBClassifier(**XGB_CLF_PARAMS, random_state=RS + fold)
        reg = XGBRegressor(**XGB_REG_PARAMS, random_state=RS + fold)
        clf.fit(Xtr_c, ytr_c, eval_set=[(Xin_c, yin_c)], verbose=False)
        reg.fit(Xtr_pos, ytr_pos, eval_set=[(Xin_pos, yin_pos)], verbose=False)

        p = clf.predict_proba(Xva)[:, 1]
        e = np.clip(reg.predict(Xva), 0, clip)
        oof[va] = np.clip(p * e, 0, clip)

    hurdle_mse = mean_squared_error(y, oof)
    hurdle_r2 = r2_score(y, oof)
    xgb_mse = robustness.query("setup == @setup_name and model == 'XGBoost'")['dev_mse'].iloc[0]
    xgb_r2  = robustness.query("setup == @setup_name and model == 'XGBoost'")['dev_r2'].iloc[0]

    print(f'Hurdle (XGBoost)   dev MSE={hurdle_mse:7.2f}  R2={hurdle_r2:.3f}')
    print(f'Single XGBoost     dev MSE={xgb_mse:7.2f}  R2={xgb_r2:.3f}')
    print(f'Delta MSE (hurdle - single) = {hurdle_mse - xgb_mse:+.2f}  '
          f'({"hurdle helps" if hurdle_mse < xgb_mse else "hurdle does not help"})')

    hurdle_rows.append({'setup': setup_name, 'target': cfg['target_label'],
                         'hurdle_mse': hurdle_mse, 'hurdle_r2': hurdle_r2,
                         'single_xgb_mse': xgb_mse, 'single_xgb_r2': xgb_r2,
                         'delta_mse': hurdle_mse - xgb_mse})


SETUP A (Q4 2025) -- hurdle model
Zero rate: 0.478


Hurdle (XGBoost)   dev MSE= 325.32  R2=0.628
Single XGBoost     dev MSE= 321.17  R2=0.633
Delta MSE (hurdle - single) = +4.15  (hurdle does not help)

SETUP C (Feb-Apr 2026) -- hurdle model
Zero rate: 0.486


Hurdle (XGBoost)   dev MSE= 306.33  R2=0.562
Single XGBoost     dev MSE= 302.77  R2=0.567
Delta MSE (hurdle - single) = +3.56  (hurdle does not help)


## 6. Summary Tables

In [7]:
ablation_df = pd.DataFrame(ablation_rows)
naive_df_summary = pd.DataFrame(naive_rows)
hurdle_df = pd.DataFrame(hurdle_rows)

print('ABLATION LADDER (5-fold CV, by setup)')
print(ablation_df.round(3).to_string(index=False))

print('\n\nNAIVE vs CLEAN TARGET (XGBoost, 5-fold CV, by setup)')
print(naive_df_summary.round(3).to_string(index=False))

print('\n\nHURDLE vs SINGLE XGBOOST (5-fold CV, by setup)')
print(hurdle_df.round(3).to_string(index=False))

ablation_df.to_csv(OUT / 'ablation_cross_period.csv', index=False)
naive_df_summary.to_csv(OUT / 'naive_vs_clean_cross_period.csv', index=False)
hurdle_df.to_csv(OUT / 'hurdle_cross_period.csv', index=False)
print('\nSaved ablation_cross_period.csv, naive_vs_clean_cross_period.csv, hurdle_cross_period.csv')

ABLATION LADDER (5-fold CV, by setup)
setup       target                       stage     mse    r2
    A      Q4 2025        A1 raw static linear 468.680 0.465
    A      Q4 2025     A2 + engineered (no TE) 407.480 0.535
    A      Q4 2025 A3 + K-fold target encoding 407.293 0.535
    A      Q4 2025             XGBoost (tuned) 321.170 0.633
    A      Q4 2025                 SLSQP blend 317.842 0.637
    C Feb-Apr 2026        A1 raw static linear 426.477 0.390
    C Feb-Apr 2026     A2 + engineered (no TE) 380.792 0.455
    C Feb-Apr 2026 A3 + K-fold target encoding 380.735 0.456
    C Feb-Apr 2026             XGBoost (tuned) 302.772 0.567
    C Feb-Apr 2026                 SLSQP blend 299.725 0.571


NAIVE vs CLEAN TARGET (XGBoost, 5-fold CV, by setup)
setup       target  clean_mse  clean_r2  naive_mse  naive_r2  delta_r2
    A      Q4 2025    321.290     0.633    314.827     0.807    -0.174
    C Feb-Apr 2026    303.753     0.566    275.317     0.831    -0.265


HURDLE vs SINGLE XGBO

## 7. Does the Ablation Story Hold Across Periods?

Ali's reference-paper ablation figure found that a clean target and deeper history dominate,
while price/review features, hyperparameter tuning, and the hurdle model each contribute
roughly zero ΔR². Section 5's hurdle comparison is a direct cross-period analogue of his
"Hurdle: +0.000" bar. Section 4's naive-vs-clean comparison is **not** a direct analogue of
his "Clean target" bar, though -- see the caveat in Section 4: R² is not comparable across
two targets with different variance, so the fair number is the MSE delta, not the R² delta.

In [8]:
print('Naive-vs-clean, by setup (neither R2 nor MSE favors clean -- see Section 4 caveat; '
      'the case for the clean target is construct validity, not a metrics comparison):')
for _, row in naive_df_summary.iterrows():
    delta_mse = row['naive_mse'] - row['clean_mse']
    print(f'  Setup {row["setup"]} ({row["target"]}): '
          f'R2 clean={row["clean_r2"]:.3f} naive={row["naive_r2"]:.3f} (not comparable)  |  '
          f'MSE delta (naive-clean)={delta_mse:+.2f}')

print('\nHurdle vs single-XGBoost delta MSE by setup (direct analogue of Ali\'s "Hurdle" ablation bar):')
for _, row in hurdle_df.iterrows():
    verdict = 'helps' if row['delta_mse'] < 0 else 'does not help'
    print(f'  Setup {row["setup"]} ({row["target"]}): {row["delta_mse"]:+.2f} MSE ({verdict})')

Naive-vs-clean, by setup (neither R2 nor MSE favors clean -- see Section 4 caveat; the case for the clean target is construct validity, not a metrics comparison):
  Setup A (Q4 2025): R2 clean=0.633 naive=0.807 (not comparable)  |  MSE delta (naive-clean)=-6.46
  Setup C (Feb-Apr 2026): R2 clean=0.566 naive=0.831 (not comparable)  |  MSE delta (naive-clean)=-28.44

Hurdle vs single-XGBoost delta MSE by setup (direct analogue of Ali's "Hurdle" ablation bar):
  Setup A (Q4 2025): +4.15 MSE (does not help)
  Setup C (Feb-Apr 2026): +3.56 MSE (does not help)


## Summary

Cross-period ablation and hurdle-model check complete for Setup A (Q4 2025) and Setup C
(Feb-Apr 2026), reusing Setup B's (Q1 2026, primary) XGBoost/blend dev-CV numbers from
`16_Cross_Period_Robustness.ipynb` rather than recomputing them. See Section 7 for whether
the clean-target finding and the hurdle model's (non-)effect generalize beyond the single
quarter (Q1 2026) they were originally established on.